# 00 · Setup (rode com internet, antes da palestra)

Baixa tudo o que os outros notebooks usam, para que eles rodem **offline**:

| O quê | Tamanho | Usado em |
|---|---|---|
| `weaviate/wiki-sample`: 50 mil passagens da Wikipedia já vetorizadas (Snowflake arctic-embed-l-v2.0, 1024 dims) | ~450 MB | 01–06 |
| `weaviate/arXiv-AI-papers-multi-vector`: 399 páginas de papers com multi-vetores ColQwen2 | ~600 MB | 04 |
| Modelo `Snowflake/snowflake-arctic-embed-l-v2.0`, para vetorizar as perguntas | ~2,2 GB | 01–03, 06 |

E sobe o Weaviate: um cluster com **3 nós** + Prometheus + Grafana, usado por todos os notebooks, nas portas padrão (REST `:8080`, gRPC `:50051`):

```sh
cd docker && docker compose up -d
```

In [1]:
import os
os.environ["HF_HUB_OFFLINE"] = "0"  # aqui sim, precisamos da rede

from huggingface_hub import hf_hub_download
import utils

for f in utils.WIKI_FILES:
    print(hf_hub_download(utils.WIKI_REPO, f, repo_type="dataset"))
print(hf_hub_download(utils.ARXIV_REPO, utils.ARXIV_FILE, repo_type="dataset"))

/Users/dudanogueira/dev/weaviate/lab/claude/latinoware-2026/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/dudanogueira/.cache/huggingface/hub/datasets--weaviate--wiki-sample/snapshots/da2f06b7aa3987ca328b6f4cab3efd6b5c6a67c2/weaviate/snowflake-arctic-v2/0001.parquet
/Users/dudanogueira/.cache/huggingface/hub/datasets--weaviate--wiki-sample/snapshots/da2f06b7aa3987ca328b6f4cab3efd6b5c6a67c2/weaviate/snowflake-arctic-v2/0002.parquet


/Users/dudanogueira/.cache/huggingface/hub/datasets--weaviate--arXiv-AI-papers-multi-vector/snapshots/58f519bea5fd02d4ba451dd432b2880723f0db91/n40_p10_images.parquet


In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(utils.MODEL_NAME)
print(model)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6965.98it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({})
)


## O modelo local reproduz os vetores do dataset?

As passagens foram vetorizadas pelo Weaviate Embeddings com o mesmo modelo. Se vetorizarmos de novo aqui, a similaridade de cosseno deve ficar perto de 1.

In [3]:
import numpy as np

wiki = utils.load_wiki(1).head(5)
local = model.encode((wiki.title + "\n" + wiki.text).tolist(), normalize_embeddings=True)
stored = np.vstack(wiki.vector.to_numpy())
stored /= np.linalg.norm(stored, axis=1, keepdims=True)
print("cosseno (local x dataset):", (local * stored).sum(axis=1).round(3))

cosseno (local x dataset): [0.955 0.96  0.981 0.971 0.969]


## Weaviate está de pé?

In [4]:
import weaviate

with weaviate.connect_to_local() as client:  # localhost:8080 + gRPC 50051
    print("versão:", client.get_meta()["version"], "| ready:", client.is_ready())
    print("nós   :", [(n.name, n.status) for n in client.cluster.nodes()])

versão: 1.40.0 | ready: True
nós   : [('weaviate-0', 'HEALTHY'), ('weaviate-1', 'HEALTHY'), ('weaviate-2', 'HEALTHY')]
